# Anreichern der OpenSky-Daten

Für die Idee in unserem Modell ein kleines Display einzubauen, welches Infos zum startenden Flugzeug anzeigt, müssen die OpenSky-Daten noch etwas aufbereitet werden. Folgende Schritte passieren:

1. Unix-Zeitstempel in Berlin-Zeit umrechnen
2. Stadt aus Flughafencode extrahieren
3. Fluggesellschaft aus Callsign extrahieren

Für Schritt 2 und 3 werden zwei frei verfügbare Tabellen geladen (OurAirports, OpenFlights).

In [33]:
from pathlib import Path
import pandas as pd

CSV_PFAD = "txl_abfluege_2019.csv"
ZEITZONE = "Europe/Berlin"

AIRPORTS_QUELLE = "https://davidmegginson.github.io/ourairports-data/airports.csv"
AIRLINES_QUELLE = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airlines.dat"

#### 0.) CSV laden
Aus dem OpenSky Network wurden folgende Daten abgerufen:
- `icao24` - Kennung des Flugzeugs
- `callsign` - Callsign des Fluges
- `firstSeen` - Unix-Zeitstempel des ersten Transpondersignals (also die ungefähre Startzeit)
- `lastSeen` - Unix-Zeitstempel des letzten Transpondersignals (also die ungefähre Landung)
- `estDepartureAirport` - ICAO-Code des Abflughafens
- `estArrivalAirport` - ICAO-Code des Zielflughafens
- `departureAirportCandidatesCount` - Anzahl der möglichen Abflughäfen
- `arrivalAirportCandidatesCount` - Anzahl der möglichen Zielflughäfen

In [34]:
df = pd.read_csv(CSV_PFAD)
df = df.sort_values("firstSeen").reset_index(drop=True) # nach Datum sortieren
print(f"{len(df):,} Abflüge")
df.head()

90,512 Abflüge


,icao24,callsign,firstSeen,lastSeen,estDepartureAirport,estArrivalAirport,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,406c72,EZY5741,1546318937,1546321925,EDDT,EPWA,0,1
1,4951c8,TAP531D,1546319032,1546331032,EDDT,LPPT,0,4
2,48418d,KLM44R,1546319631,1546323529,EDDT,EHAM,0,1
3,4cabb9,LDM14HL,1546319900,1546333161,EDDT,NaN,0,0
4,400cd6,EZY69FB,1546320468,1546323616,EDDT,EDDM,0,4


#### 1.) Zeitstempel umrechnen
Die Zeitstempel, welche das OpenSky-Network liefert, sind in Unixzeit. Diese werden in Berliner Zeit umgerechnet und dann die Flugdauer berechnet.

In [35]:
df["abflug"]  = pd.to_datetime(df["firstSeen"], unit="s", utc=True).dt.tz_convert(ZEITZONE)
df["ankunft"] = pd.to_datetime(df["lastSeen"],  unit="s", utc=True).dt.tz_convert(ZEITZONE)
df["flugdauer_min"] = ((df["lastSeen"] - df["firstSeen"]) / 60).round().astype(int)
df[["callsign", "abflug", "ankunft", "flugdauer_min"]].head()

,callsign,abflug,ankunft,flugdauer_min
0,EZY5741,2019-01-01 06:02:17+01:00,2019-01-01 06:52:05+01:00,50
1,TAP531D,2019-01-01 06:03:52+01:00,2019-01-01 09:23:52+01:00,200
2,KLM44R,2019-01-01 06:13:51+01:00,2019-01-01 07:18:49+01:00,65
3,LDM14HL,2019-01-01 06:18:20+01:00,2019-01-01 09:59:21+01:00,221
4,EZY69FB,2019-01-01 06:27:48+01:00,2019-01-01 07:20:16+01:00,52


#### 2.) Stadt aus Flughafencode extrahieren

In [36]:
ap = pd.read_csv(AIRPORTS_QUELLE, low_memory=False)
ap["icao"] = ap["icao_code"].fillna(ap["gps_code"]).fillna(ap["ident"])
# Flughäfen mit Linienverkehr zuerst, damit bei doppelten Codes der richtige bleibt
ap = ap.sort_values("scheduled_service", ascending=False).drop_duplicates("icao").set_index("icao")

code_ziel = df["estArrivalAirport"]
df["ziel_flughafen"] = code_ziel.map(ap["name"])
df["ziel_stadt"]     = code_ziel.map(ap["municipality"]).fillna(df["ziel_flughafen"])
df["ziel_land"]      = code_ziel.map(ap["iso_country"])
df["ziel_stadt"]     = df["ziel_stadt"].fillna("unbekannt")

print("Anzahl der Flüge mit unbekanntem Ziel:", df["estArrivalAirport"].isna().sum())
print("Anzahl der Flüge deren Flughafencode nicht in OurAirports gefunden wurde:", (df["estArrivalAirport"].notna() & df["ziel_flughafen"].isna()).sum())

df["ziel_stadt"].value_counts().head(7)

Anzahl der Flüge mit unbekanntem Ziel: 9648
Anzahl der Flüge deren Flughafencode nicht in OurAirports gefunden wurde: 388


ziel_stadt
unbekannt            10036
Munich                8262
Frankfurt am Main     7966
Köln (Cologne)        5865
Stuttgart             5329
Düsseldorf            5318
Zurich                4734
Name: count, dtype: int64

#### 3.) Fluggesellschaft aus Callsign extrahieren

Das Callsing bei den meisten Flügen besteht aus dem 3-stelligen ICAO-Airline-Code plus Flugnummer (`EZY72PK` → `EZY` = easyJet).
Die häufigsten nicht gefundenen Codes wurden von https://iata.codes/airlines/ ergänzt.

In [37]:
al = pd.read_csv(
    AIRLINES_QUELLE, header=None, na_values=["\\N", "N/A", "-"],
    names=["id", "name", "alias", "iata", "icao", "callsign_name", "land", "aktiv"],
)
al = al.dropna(subset=["icao"]).sort_values("aktiv", ascending=False).drop_duplicates("icao").set_index("icao")

# ergänzte Codes
AIRLINE_KORREKTUREN = {
    "EZY": "easyJet",
    "EJU": "easyJet Europe",
    "EWG": "Eurowings",
    "DLH": "Lufthansa",
    "RYR": "Ryanair",
    "TUI": "TUIfly",
    "BEL": "Brussels Airlines",
    "LDM": "Lauda",
    "EZS": "Easyjet Switzerland"
}

df["callsign"] = df["callsign"].fillna("").str.strip()
df["airline_code"] = df["callsign"].str.extract(r"^([A-Z]{3})\d", expand=False)

df["airline"] = (
    df["airline_code"].map(AIRLINE_KORREKTUREN)
    .fillna(df["airline_code"].map(al["name"]))
    .fillna(df["airline_code"])
    .fillna("unbekannt")
)
df["airline"].value_counts().head(7)

airline
easyJet Europe                   20057
Eurowings                        13994
Lufthansa                        11623
easyJet                           7478
Swiss International Air Lines     3310
Ryanair                           3192
Austrian Airlines                 2570
Name: count, dtype: int64

In [38]:
# Welche Codes konnten nicht gefunden werden?
offen = df[df["airline_code"].notna()
           & ~df["airline_code"].isin(al.index)
           & ~df["airline_code"].isin(AIRLINE_KORREKTUREN)]
offen["airline_code"].value_counts().head(7)

airline_code
CHX    92
NJE    68
BFD    51
SXD    36
TWI    31
OTF    24
JKH    10
Name: count, dtype: int64

In [39]:
df.head()

,icao24,callsign,firstSeen,lastSeen,estDepartureAirport,estArrivalAirport,departureAirportCandidatesCount,arrivalAirportCandidatesCount,abflug,ankunft,flugdauer_min,ziel_flughafen,ziel_stadt,ziel_land,airline_code,airline
0,406c72,EZY5741,1546318937,1546321925,EDDT,EPWA,0,1,2019-01-01 06:02:17+01:00,2019-01-01 06:52:05+01:00,50,Warsaw Chopin Airport,Warsaw,PL,EZY,easyJet
1,4951c8,TAP531D,1546319032,1546331032,EDDT,LPPT,0,4,2019-01-01 06:03:52+01:00,2019-01-01 09:23:52+01:00,200,Lisbon Humberto Delgado Airport,Lisbon,PT,TAP,TAP Portugal
2,48418d,KLM44R,1546319631,1546323529,EDDT,EHAM,0,1,2019-01-01 06:13:51+01:00,2019-01-01 07:18:49+01:00,65,Amsterdam Airport Schiphol,Amsterdam,NL,KLM,KLM Royal Dutch Airlines
3,4cabb9,LDM14HL,1546319900,1546333161,EDDT,NaN,0,0,2019-01-01 06:18:20+01:00,2019-01-01 09:59:21+01:00,221,NaN,unbekannt,NaN,LDM,Lauda
4,400cd6,EZY69FB,1546320468,1546323616,EDDT,EDDM,0,4,2019-01-01 06:27:48+01:00,2019-01-01 07:20:16+01:00,52,Munich Airport,Munich,DE,EZY,easyJet


#### 4.) als CSV speichern

In [40]:
# zu speichernde Spalten
spalten = ['icao24','callsign','abflug','ankunft','flugdauer_min','ziel_flughafen','ziel_stadt','ziel_land','airline']
# fehlende Werte zu "unbekannt" machen
df[spalten] = df[spalten].fillna("unbekannt")

# speichern
df.to_csv("txl_abfluege_2019_infos.csv", columns=spalten, index=False)
print(f"CSV gespeichert: {Path('txl_abfluege_2019_infos.csv').absolute()}")

CSV gespeichert: /Users/tillman/PycharmProjects/flugdaten_2019_eddt/txl_abfluege_2019_infos.csv
